# SemMap: проверка H1 на ConceptNet-10k с разделением типов отношений

Исходная сеть: ровно 10 000 понятий ConceptNet, отобранных воспроизводимым relation-stratified обходом. Направленные отношения IsA, RelatedTo, PartOf и остальные заданные типы сохраняются раздельными слоями. Весь тест выполняется на одной multilayer nSBM; далее выводятся показатели по типам отношений.

Исправлено: утечка чётности индексов в искусственном генераторе; сравнение истинной и найденной иерархии через ARI; диагностика согласованности инициализаций; Hits@10 только при >10 кандидатах.

Исполнение: до 12 доступных CPU-ядер для graph-tool / OpenMP, CUDA для независимой пакетной W-only блочной постобработки. GPU не выполняет обучение nSBM.

Для данных необходим исходный 5-колоночный файл assertion TSV/TSV.GZ ConceptNet, не агрегированный selected_edges.jsonl. Всё вычисляется локально в Colab и сохраняется по контрольным точкам в Google Drive.

QUICK проверяет контур, VALIDATION и FULL — оба нулевых ансамбля с более высокой разрешающей способностью p, при условии отдельной научной калибровки.

In [ ]:
# ===== ЕДИНЫЕ ПАРАМЕТРЫ =====
REPOSITORY_URL = "https://github.com/SemanticMap/semgraphex.git"
REPO_BRANCH = "feature/h1-cn10k-relation-aware-validation"
SOURCE_REVISION = "a93657880552c6d20656c3b3f737bc41b89bf679"
REPO_DIR = "/content/semmap_h1_cn10k_repo"
RUN_PROFILE = "QUICK"             # QUICK | VALIDATION | FULL; новый профиль => новый RUN_ID
RUN_ID = "h1-cn10k-typed-quick-01"
SEED = 42
TARGET_NODES = 10000
LANGUAGE = "en"
RELATIONS = ("RelatedTo", "IsA", "PartOf", "UsedFor",
             "HasProperty", "HasA", "Synonym", "Causes")
MIN_WEIGHT = 0.0
MAX_QUALIFYING_ASSERTIONS = 2_000_000  # защита от OOM, НЕ префиксная выборка
CPU_BUDGET = 12
REQUIRE_GPU = True
GPU_BATCH_SIZE = 150_000
GPU_MAX_LEVELS = 5
CHECKPOINT_INTERVAL_SECONDS = 90
USE_DRIVE = True
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/SemanticMap"
# Истинный файл ConceptNet 5-column assertion TSV/TSV.GZ
CONCEPTNET_TSV_ON_DRIVE = ""
RUN_CALIBRATION = False
CALIBRATION_SIMULATIONS = 10
CALIBRATION_RESTARTS = 3
CALIBRATION_NULLS = 49
PROFILES = {
    "QUICK": {"restarts": 2, "nulls": 3, "negatives": 100},
    "VALIDATION": {"restarts": 3, "nulls": 49, "negatives": 100},
    "FULL": {"restarts": 10, "nulls": 99, "negatives": 100},
}
assert RUN_PROFILE in PROFILES
assert TARGET_NODES == 10000 and len(set(RELATIONS)) == len(RELATIONS)
print(RUN_PROFILE, PROFILES[RUN_PROFILE], "relations=", RELATIONS)

In [ ]:
# 1. Проверка реальных ресурсов: ядра (включая cgroup), RAM, диск, CUDA
from pathlib import Path
import os, sys, shutil, subprocess, json, time, platform, hashlib

def usable_cpus():
    count = os.cpu_count() or 1
    if hasattr(os, "sched_getaffinity"):
        count = min(count, len(os.sched_getaffinity(0)))
    cpu_max = Path("/sys/fs/cgroup/cpu.max")
    if cpu_max.exists():
        quota, period = cpu_max.read_text().split()
        if quota != "max":
            count = min(count, max(1, -(-int(quota) // int(period))))
    return max(1, count)

AVAILABLE_CPU = usable_cpus()
ALLOTTED_CPU = min(CPU_BUDGET, AVAILABLE_CPU)
FREE_DISK_GB = shutil.disk_usage("/content").free / (1024 ** 3)
ram_gb = (os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")) / 1024**3
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
GPU_VISIBLE = gpu.returncode == 0 and "GPU" in gpu.stdout
print({"python": sys.version.split()[0], "os": platform.platform(),
       "available_cpus": AVAILABLE_CPU, "requested_budget": CPU_BUDGET,
       "effective_cpu_budget": ALLOTTED_CPU,
       "free_disk_gb": round(FREE_DISK_GB, 1),
       "host_available_ram_gb": round(ram_gb, 1),
       "cuda_gpu": gpu.stdout.strip() if GPU_VISIBLE else "not detected"})
if ALLOTTED_CPU < CPU_BUDGET:
    print(f"ВНИМАНИЕ: вместо запрошенных {CPU_BUDGET} доступно {ALLOTTED_CPU}; не обещаем отсутствующие ядра.")
if FREE_DISK_GB < 7:
    raise RuntimeError("Для установки graph-tool требуется как минимум ~7 GiB свободного /content")
if REQUIRE_GPU and not GPU_VISIBLE:
    raise RuntimeError("GPU не обнаружен: выберите GPU в настройках Colab либо REQUIRE_GPU=False")

In [ ]:
# 2. Google Drive: только долговременные данные и контрольные точки
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path(DRIVE_PROJECT_ROOT)
WORK_ROOT = Path("/content/semmap_h1_work")
BIN_ROOT = Path("/content/semmap_h1_bin")
ENV_ROOT = Path("/content/semmap_h1_env")
REPOSITORY = Path(REPO_DIR)
for directory in (WORK_ROOT, BIN_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
RUN_LOCAL = WORK_ROOT / "runs" / RUN_ID
RUN_DRIVE = DRIVE_ROOT / "colab" / "hypothesis" / "runs" / RUN_ID
if USE_DRIVE:
    if not Path("/content/drive").is_mount():
        raise RuntimeError("Google Drive не смонтирован")
    RUN_DRIVE.mkdir(parents=True, exist_ok=True)
print("Локальная рабочая папка:", RUN_LOCAL)
print("Постоянное хранилище:", RUN_DRIVE if USE_DRIVE else "(отключено; файлы могут быть потеряны)")

In [ ]:
# 3. Git: получаем код из репозитория и фиксируем именно указанную ревизию
if not (REPOSITORY / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "75", "--branch",
                    REPO_BRANCH, REPOSITORY_URL,
                    str(REPOSITORY)], check=True)
else:
    subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "--depth", "75",
                    "origin", REPO_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "checkout", "--detach",
                SOURCE_REVISION], check=True)
checked = subprocess.check_output(["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], text=True).strip()
assert checked == SOURCE_REVISION
print("Закреплён код semgraphex:", checked)

## 4. Graph-tool без несовместимости Python

Colab и системный Ubuntu Python могут иметь разные версии. Поэтому используем **изолированное окружение micromamba** с graph-tool-base 2.98 и Python 3.12. Не заменяем Python работающего блокнота; все вычисления nSBM идут отдельным процессом окружения.

Пакеты устанавливаются из conda-forge. После установки ноутбук **обязательно проверяет реальный импорт graph-tool**, а не скрывает ошибку или переключает алгоритм на другой. Пакеты не обновляются без необходимости в базовом Colab. Для повторяемости сохраняется список установленных версий.

In [ ]:
# 4. Установка micromamba и рабочего окружения graph-tool (один раз на новый runtime)
MAMBA = BIN_ROOT / "bin" / "micromamba"
ENV_PY = ENV_ROOT / "bin" / "python"
BOOTSTRAP_LOG = WORK_ROOT / "bootstrap.log"

def checked_log(command, *, cwd=None, env=None, log_path=BOOTSTRAP_LOG):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(map(str, command)) + "\n")
        log.flush()
        outcome = subprocess.run(command, cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT)
    if outcome.returncode:
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-45:]
        raise RuntimeError(f"Команда завершилась с ошибкой {outcome.returncode}:\n" + "\n".join(tail))
    return outcome

if not MAMBA.exists():
    BIN_ROOT.mkdir(parents=True, exist_ok=True)
    checked_log(["bash", "-lc", "set -o pipefail; curl -fsSL "
                 "https://micro.mamba.pm/api/micromamba/linux-64/latest "
                 "| tar -xj -C " + str(BIN_ROOT) + " bin/micromamba"])
mamba_env = dict(os.environ, MAMBA_ROOT_PREFIX="/content/semmap_mamba")
if not ENV_PY.exists():
    checked_log([str(MAMBA), "create", "-y", "-p", str(ENV_ROOT),
                 "-c", "conda-forge", "--strict-channel-priority",
                 "python=3.12", "graph-tool-base=2.98", "numpy",
                 "scipy", "scikit-learn", "pyyaml", "networkx",
                 "pytest", "psutil", "threadpoolctl"], env=mamba_env)
version = subprocess.check_output(
    [str(ENV_PY), "-c",
     "import graph_tool.all as gt; import numpy; print(gt.__version__,numpy.__version__)"],
    text=True, stderr=subprocess.STDOUT,
).strip()
print("Исполняемое окружение:", ENV_PY, "\n", version)
CONDA_SPEC = WORK_ROOT / "environment_packages.json"
CONDA_SPEC.write_text(subprocess.check_output(
    [str(MAMBA), "list", "-p", str(ENV_ROOT), "--json"],
    env=mamba_env, text=True), encoding="utf-8")
if USE_DRIVE:
    copy_target = RUN_DRIVE / "environment_packages.json"
    if copy_target.exists() and copy_target.read_bytes() != CONDA_SPEC.read_bytes():
        print("ВНИМАНИЕ: окружение отличается от предыдущего запуска; см. версии библиотек.")
    else:
        copy_if_environment = copy_target.with_suffix(".part")
        shutil.copy2(CONDA_SPEC, copy_if_environment)
        os.replace(copy_if_environment, copy_target)

In [ ]:
# 5. Реальные интеграционные тесты graph-tool (до долгого эксперимента)
PYTHONPATH = str(REPOSITORY / "src") + ":" + str(REPOSITORY)
CPU_ENV = dict(os.environ, PYTHONPATH=PYTHONPATH, OPENBLAS_NUM_THREADS="1",
               MKL_NUM_THREADS="1", NUMEXPR_NUM_THREADS="1", OMP_NUM_THREADS="1",
               PYTHONUNBUFFERED="1")
checked_log([str(ENV_PY), "-m", "pytest", "-q",
             "tests/test_h1_graph_tool_integration.py",
             "tests/test_h1_science.py",
             "tests/test_h1_runner.py",
             "tests/test_h1_cn10k_relations.py"],
            cwd=REPOSITORY, env=CPU_ENV, log_path=WORK_ROOT / "h1_preflight_tests.log")
print("Интеграционные тесты graph-tool прошли в этом runtime.")

In [ ]:
# 6. GPU-first: CuPy только для проекции W. Ранний CUDA/CPU smoke.
import numpy as np
sys.path.insert(0, str(REPOSITORY / "src"))
from semmap_haken.hypothesis.graph import TypedGraph
from semmap_haken.hypothesis.projection import estimate_block_kernel

if GPU_VISIBLE:
    try:
        import cupy as cp
    except ImportError:
        # Colab L4 с CUDA 12.x; для иной основной версии CUDA нужен другой wheel.
        checked_log([sys.executable, "-m", "pip", "install", "--quiet", "cupy-cuda12x>=13,<15"],
                    log_path=WORK_ROOT / "cupy_install.log")
        import cupy as cp
    assert cp.cuda.runtime.getDeviceCount() > 0
    print("CUDA device:", cp.cuda.runtime.getDeviceProperties(0)["name"])
smoke_g = TypedGraph(
    ("a", "b", "c", "d"), ("/r/IsA",),
    np.array([0, 0, 1, 2]), np.array([1, 2, 2, 3]),
    np.array([0, 0, 0, 0], dtype=np.int32),
    np.ones(4), ("a1", "a2", "a3", "a4"))
smoke_p = np.array([0, 0, 1, 1], dtype=np.int64)
ref = estimate_block_kernel(smoke_g, smoke_p, device="cpu", batch_size=2)
if GPU_VISIBLE:
    accel = estimate_block_kernel(smoke_g, smoke_p, device="cuda", batch_size=2)
    assert accel["backend"] == "cuda"
    assert [(v["source_block"],v["target_block"],v["edge_count"])
            for v in accel["entries"]] == [
        (v["source_block"],v["target_block"],v["edge_count"]) for v in ref["entries"]]
    assert np.allclose([v["weight_sum"] for v in ref["entries"]],
                       [v["weight_sum"] for v in accel["entries"]])
    print("GPU W-проекция: CUDA / CPU проверка совпадения пройдена.")
else:
    print("GPU нет: проекция W на CPU (в конфигурации REQUIRE_GPU=False)")

In [ ]:
# 7. Сборка репрезентативного по отношениям (но НЕ равномерного) ConceptNet-10k
import json
STAGED_INPUT = WORK_ROOT / "inputs" / RUN_ID
STAGED_INPUT.mkdir(parents=True, exist_ok=True)
if not CONCEPTNET_TSV_ON_DRIVE:
    raise ValueError("Заполните CONCEPTNET_TSV_ON_DRIVE: необходим 5-колоночный исходный ConceptNet TSV/TSV.GZ")
source = Path(CONCEPTNET_TSV_ON_DRIVE)
if not source.is_file():
    raise FileNotFoundError(source)
if source.name.endswith(".jsonl") or not (
    source.name.endswith(".tsv") or source.name.endswith(".tsv.gz")
    or source.name.endswith(".tsv.bz2") or source.name.endswith(".gz")
):
    raise ValueError("Нужен исходный TSV/TSV.GZ, НЕ JSONL/NPZ")

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as reader:
        for chunk in iter(lambda: reader.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

RAW_LOCAL = STAGED_INPUT / source.name
if not RAW_LOCAL.exists() or RAW_LOCAL.stat().st_size != source.stat().st_size:
    shutil.copy2(source, RAW_LOCAL)
if file_hash(source) != file_hash(RAW_LOCAL):
    raise RuntimeError("Копия источника на /content отличается от файла Drive")
RAW_SHA = file_hash(RAW_LOCAL)

SAMPLE_LOCAL = STAGED_INPUT / "conceptnet_en_10k_relations.tsv"
SAMPLE_META = STAGED_INPUT / (SAMPLE_LOCAL.name + ".selection.json")
SAMPLE_DRIVE = DRIVE_ROOT / "colab" / "hypothesis" / "inputs" / RUN_ID
if USE_DRIVE:
    SAMPLE_DRIVE.mkdir(parents=True, exist_ok=True)
    remote_file = SAMPLE_DRIVE / SAMPLE_LOCAL.name
    remote_meta = SAMPLE_DRIVE / SAMPLE_META.name
    if not SAMPLE_LOCAL.exists() and remote_file.exists() and remote_meta.exists():
        shutil.copy2(remote_file, SAMPLE_LOCAL)
        shutil.copy2(remote_meta, SAMPLE_META)

if SAMPLE_LOCAL.exists():
    if not SAMPLE_META.exists():
        raise RuntimeError("Обнаружена выборка без selection.json")
    SAMPLE_REPORT = json.loads(SAMPLE_META.read_text(encoding="utf-8"))
    compatible = (
        SAMPLE_REPORT.get("source_sha256") == RAW_SHA
        and SAMPLE_REPORT.get("relations_requested") == list(RELATIONS)
        and SAMPLE_REPORT.get("seed") == SEED
        and SAMPLE_REPORT.get("selected_nodes") == TARGET_NODES
        and SAMPLE_REPORT.get("language") == LANGUAGE
        and SAMPLE_REPORT.get("min_weight") == float(MIN_WEIGHT)
        and SAMPLE_REPORT.get("sha256") == file_hash(SAMPLE_LOCAL)
    )
    if not compatible:
        raise ValueError("Старая выборка несовместима с конфигурацией: нужен новый RUN_ID")
else:
    args = [str(ENV_PY), "-m", "semmap_haken.hypothesis.conceptnet_10k",
            "--input", str(RAW_LOCAL), "--output", str(SAMPLE_LOCAL),
            "--nodes", str(TARGET_NODES), "--relations", ",".join(RELATIONS),
            "--language", LANGUAGE, "--seed", str(SEED),
            "--max-assertions", str(MAX_QUALIFYING_ASSERTIONS),
            "--min-weight", str(MIN_WEIGHT)]
    checked_log(args, cwd=REPOSITORY, env=CPU_ENV,
                log_path=WORK_ROOT / "cn10k_selection.log")
    SAMPLE_REPORT = json.loads(SAMPLE_META.read_text(encoding="utf-8"))
    if USE_DRIVE:
        for src in (SAMPLE_LOCAL, SAMPLE_META):
            dest = SAMPLE_DRIVE / src.name
            tmp = dest.with_name(dest.name + ".partial")
            shutil.copy2(src, tmp)
            os.replace(tmp, dest)

assert SAMPLE_REPORT["selected_nodes"] == TARGET_NODES
assert sum(SAMPLE_REPORT["by_relation"].values()) == SAMPLE_REPORT["selected_assertions"]
INPUT_LOCAL = SAMPLE_LOCAL
DATA_SHA = file_hash(INPUT_LOCAL)
print("Выборка:", SAMPLE_REPORT["selected_nodes"], "узлов,",
      SAMPLE_REPORT["selected_assertions"], "утверждений")
for relation, count in SAMPLE_REPORT["by_relation"].items():
    print(f"  {relation:<19}: {count:>9,d}")
print("SHA-256:", DATA_SHA)
print("ВНИМАНИЕ:", SAMPLE_REPORT["sampling_warning"])

## 8. Контрольные точки и восстановление

Все повторные SBM/DC-SBM/nSBM сохраняются как независимые JSON/NPZ. Снимки задач шардируются по контрольным графам в Google Drive.

Выборка ConceptNet из 10 000 понятий, её SHA-256, список отношений и протокол отбора сохраняются отдельно: MyDrive/SemanticMap/colab/hypothesis/inputs/<RUN_ID>.

Сначала H1 выполняется на быстром локальном диске Colab. После сбоя сеанса открывайте тот же ноутбук с тем же RUN_ID, параметрами, входом и закреплённой версией кода — RESUME восстановит готовые задания. Для иных типов отношений или размеров выборки создавайте новый RUN_ID.

In [ ]:
# 8. Минимальный менеджер снимков: изменённые файлы -> Drive, не все графы.
def checkpoint_group(task_name):
    # real-* | null_configuration_0000-* | null_conditional_flat_dc_sbm_0000-*
    return task_name.split("-", 1)[0]

def copy_if_new(src, dst):
    if not src.is_file():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return False  # записи задач неизменяемы, каждый RUN_ID уникален
    partial = dst.with_name(dst.name + ".part")
    shutil.copy2(src, partial)
    os.replace(partial, dst)
    return True

def restore_run(local_run, drive_run):
    """Restore checkpoint shards to the flat tasks directory expected by runner."""
    if not USE_DRIVE or not (drive_run / "manifest.json").is_file():
        return False
    if local_run.exists() and any(local_run.iterdir()):
        raise RuntimeError(f"Локальный запуск уже содержит данные: {local_run}")
    copy_if_new(drive_run / "manifest.json", local_run / "manifest.json")
    for root in ("nulls", "gpu_projection"):
        directory = drive_run / root
        if directory.exists():
            for src in directory.rglob("*"):
                if src.is_file() and not src.name.endswith(".part"):
                    copy_if_new(src, local_run / root / src.relative_to(directory))
    checkpoint_dir = drive_run / "checkpoints"
    if checkpoint_dir.exists():
        for src in checkpoint_dir.rglob("*"):
            if src.is_file() and src.suffix in {".json", ".npz"}:
                copy_if_new(src, local_run / "tasks" / src.name)
    for name in ("report.json", "COMPLETED"):
        copy_if_new(drive_run / name, local_run / name)
    return True

def persist_run(local_run, drive_run):
    """Sharded Drive snapshots; never persist ephemeral graph data/ subfolders."""
    if not USE_DRIVE or not local_run.exists():
        return 0
    copies = copy_if_new(local_run / "manifest.json", drive_run / "manifest.json")
    for root in ("tasks", "nulls", "gpu_projection"):
        source_dir = local_run / root
        if not source_dir.exists():
            continue
        for src in source_dir.rglob("*"):
            if not src.is_file() or src.name.endswith((".part", ".tmp")):
                continue
            rel = src.relative_to(source_dir)
            target = (drive_run / "checkpoints" / checkpoint_group(src.stem) / src.name
                      if root == "tasks" else drive_run / root / rel)
            copies += copy_if_new(src, target)
    copies += copy_if_new(local_run / "report.json", drive_run / "report.json")
    copies += copy_if_new(local_run / "COMPLETED", drive_run / "COMPLETED")
    return copies

def restore_checkpoints():
    return restore_run(RUN_LOCAL, RUN_DRIVE)

def persist_checkpoints():
    return persist_run(RUN_LOCAL, RUN_DRIVE)

restored = restore_checkpoints() if USE_DRIVE and not RUN_LOCAL.exists() else False
MODE = "RESUME" if (RUN_LOCAL / "manifest.json").exists() else "NEW"
print({"restored_from_drive": restored, "mode": MODE, "run_id": RUN_ID})

In [ ]:
# 9. H1: сравнение моделей и оба типа нулевых графов.
params = PROFILES[RUN_PROFILE]
RESTARTS, NULLS = params["restarts"], params["nulls"]
TOTAL_FITS = 3 * RESTARTS * (1 + 2 * NULLS)
print("Модельных подгонок:", TOTAL_FITS)
print("Многореляционный граф: ", len(RELATIONS), "типов связей")
if RUN_PROFILE == "QUICK":
    print("QUICK: 3 контроля каждого типа => научная p-значимость недостижима.")
elif RUN_PROFILE == "VALIDATION":
    print("VALIDATION: 49 контролей каждого ансамбля; нужна калибровка мощности.")
elif RUN_PROFILE == "FULL":
    print("FULL: 99 контролей каждого ансамбля, множество перезапусков.")
H1_ARGS = [str(ENV_PY), "-m", "semmap_haken.hypothesis.h1_cli",
           "--input", str(INPUT_LOCAL), "--output", str(RUN_LOCAL),
           "--mode", MODE, "--seed", str(SEED), "--restarts", str(RESTARTS),
           "--nulls", str(NULLS), "--workers", "auto",
           "--negatives", str(params["negatives"]), "--bootstrap", "500"]
CPU_IDS = sorted(os.sched_getaffinity(0))[:ALLOTTED_CPU] if hasattr(os, "sched_getaffinity") else []
if CPU_IDS and shutil.which("taskset"):
    H1_COMMAND = ["taskset", "-c", ",".join(map(str, CPU_IDS))] + H1_ARGS
else:
    if ALLOTTED_CPU < AVAILABLE_CPU:
        raise RuntimeError("Не найден taskset для ограничения нагрузки до заданного CPU бюджета")
    H1_COMMAND = H1_ARGS
print("CPU/OpenMP:", ALLOTTED_CPU, "доступных ядер; ускоритель CUDA используется отдельно.")

In [ ]:
# 10. Запуск: периодическое резервирование задач в Drive, корректное RESUME
LOGS = WORK_ROOT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
MAIN_LOG = LOGS / f"{RUN_ID}.log"

def last_log_lines(path, n=4):
    if not path.exists():
        return []
    with path.open("rb") as f:
        f.seek(max(0, f.seek(0, 2) - 5000))
        return f.read().decode("utf-8", errors="replace").splitlines()[-n:]

if (RUN_LOCAL / "COMPLETED").exists() and (RUN_LOCAL / "report.json").exists():
    print("Этот RUN_ID уже завершён: повторное обучение не запускается.")
else:
    with MAIN_LOG.open("a", encoding="utf-8") as log:
        log.write(f"\n=== START {time.strftime('%Y-%m-%d %H:%M:%S')} {MODE} ===\n")
        log.flush()
        process = subprocess.Popen(H1_COMMAND, cwd=REPOSITORY, env=CPU_ENV,
                                   stdout=log, stderr=subprocess.STDOUT)
        try:
            while process.poll() is None:
                time.sleep(CHECKPOINT_INTERVAL_SECONDS)
                n_synced = persist_checkpoints()
                done = len(list((RUN_LOCAL / "tasks").glob("*.json"))) if (RUN_LOCAL / "tasks").exists() else 0
                print(f"Готово модельных задач: {done}/{TOTAL_FITS}; обновлено файлов на Drive: {n_synced}")
            return_code = process.wait()
        except KeyboardInterrupt:
            print("Прерывание: останавливаем процесс, сохраняем готовые контрольные точки.")
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            persist_checkpoints()
            raise
        finally:
            n_synced = persist_checkpoints()
    print("Завершение процесса:", return_code, "| файлов синхронизировано:", n_synced)
    if return_code:
        print("\n".join(last_log_lines(MAIN_LOG, 25)))
        raise RuntimeError("H1 завершился с ошибкой; исправьте причину и повторите RUN_ID с RESUME")
assert (RUN_LOCAL / "report.json").is_file(), "Нет отчёта H1"
assert (RUN_LOCAL / "COMPLETED").is_file(), "Нет маркера COMPLETE: эксперимент не завершён"
persist_checkpoints()

In [ ]:
# 11. MDL, согласованность nSBM между запусками, точность по отношениям
REPORT = json.loads((RUN_LOCAL / "report.json").read_text(encoding="utf-8"))
print("НАУЧНЫЙ СТАТУС:", REPORT["status"], "|", REPORT["decision"].get("reason"))
print("Объём данных / разбиение:", REPORT["counts"])
print("MDL выигрыш nSBM относительно DC-SBM:", REPORT["observed_mdl_gain_bits"], "бит")
for kind, info in REPORT["fits"].items():
    print(kind, "| MDL:", round(info["best_mdl_bits"], 2),
          "| размеры блоков:", info["blocks_by_level"],
          "| диапазон MDL:", info["range_mdl_bits"])
stability = REPORT["fits"]["nested-DC-SBM"].get("restart_stability")
print("Согласованность найденных иерархий между независимыми инициализациями (ARI):", stability)
for name, null in REPORT["nulls"].items():
    print("Контроль:", name, "| p:", null["significance"])
print("\nСлои отношений (общая направленная типизированная nSBM):")
metric = REPORT["prediction"]["nested-DC-SBM"]["test"].get("by_relation", {})
for relation, volumes in REPORT["relations"].items():
    score = metric.get(relation, {})
    print(f"{relation:<16} edges={volumes['full']:>7,d}, "
          f"train={volumes['train']:>7,d}, test={volumes['test']:>5,d}, "
          f"MRR={score.get('mrr')}, H@10={score.get('hits_at_10')}, "
          f"H@10 queries={score.get('hits_at_10_queries')}")
print("Все значения MRR/H@10 относятся к заданной выборке кандидатов, не ко всему словарю понятий.")

In [ ]:
# 11a. По типам отношений: количество, MRR, контрольные MDL
import matplotlib.pyplot as plt
relations = list(REPORT["relations"])
source_counts = [REPORT["relations"][r]["full"] for r in relations]
test_counts = [REPORT["relations"][r]["test"] for r in relations]
pos = np.arange(len(relations))
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(pos - .18, source_counts, width=.36, label="Исходные утверждения")
ax.bar(pos + .18, test_counts, width=.36, label="Тестовые утверждения")
ax.set_xticks(pos, relations, rotation=35, ha="right")
ax.set(ylabel="Рёбра", title="Состав ConceptNet-10k по типам отношений")
ax.legend()
plt.tight_layout()
plt.show()

scores = REPORT["prediction"]["nested-DC-SBM"]["test"].get("by_relation", {})
r = [r for r in relations if scores.get(r, {}).get("mrr") is not None]
if r:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(r, [scores[name]["mrr"] for name in r])
    ax.set(ylim=(0, 1), ylabel="MRR (по выбранным кандидатам)",
           title="Качество nSBM по типам отношений")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

null_arrays = {name: d["mdlgain_controls"] for name, d in REPORT["nulls"].items()
               if d["mdlgain_controls"]}
if null_arrays:
    fig, ax = plt.subplots(figsize=(9, 4))
    for name, arr in null_arrays.items():
        ax.hist(arr, bins=min(18, max(4, len(arr)//3)), alpha=.55, label=name)
    ax.axvline(REPORT["observed_mdl_gain_bits"], linestyle="--",
               linewidth=2, label="ConceptNet-10k")
    ax.set(xlabel="Величина MDL выигрыша nSBM (бит)",
           ylabel="Количество контрольных графов", title="Два контрольных ансамбля")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 12. CUDA-проекция W по найденным вложенным блокам; точность сравниваем с CPU.
# Это постобработка графонной структуры и НЕ часть H1 MDL-прогона.
if GPU_VISIBLE:
    from semmap_haken.hypothesis.graph import TypedGraph
    from semmap_haken.hypothesis.projection import estimate_block_kernel
    train_folder = RUN_LOCAL / "data" / "train"
    meta = json.loads((train_folder / "ids.json").read_text(encoding="utf-8"))
    training = TypedGraph(
        tuple(meta["node_ids"]), tuple(meta["relation_names"]),
        *(np.load(train_folder / f"{name}.npy", mmap_mode="r")
          for name in ("source", "target", "relation", "weight")),
        tuple(meta["assertion_ids"]))
    task_files = sorted((RUN_LOCAL / "tasks").glob("real-nested_DC_SBM-restart_*.json"))
    if not task_files:
        raise RuntimeError("Не найдено ни одной реальной nSBM-оптимизации")
    winning = min(task_files, key=lambda p: json.loads(p.read_text())["mdl_bits"])
    with np.load(winning.with_suffix(".npz"), allow_pickle=False) as saved:
        labels = [saved[f"p{i}"].copy() for i in range(len(saved.files))]
    # Независимая CPU / CUDA проверка на фиксированном подмножестве обучающих рёбер
    subset = training.edge_subset(np.arange(training.edge_count) < min(20_000, training.edge_count))
    cpu = estimate_block_kernel(subset, labels[0], device="cpu", batch_size=GPU_BATCH_SIZE)
    cuda = estimate_block_kernel(subset, labels[0], device="cuda", batch_size=GPU_BATCH_SIZE)
    def numeric_entries(result):
        return [(v["relation"],v["source_block"],v["target_block"],
                 v["edge_count"],v["weight_sum"]) for v in result["entries"]]
    a, b = numeric_entries(cpu), numeric_entries(cuda)
    assert len(a) == len(b)
    for v, w in zip(a, b):
        assert v[:4] == w[:4] and np.isclose(v[4], w[4], rtol=1e-9, atol=1e-7)
    print("Проверено совпадение CPU и CUDA на", subset.edge_count, "рёбрах.")
    outputs = RUN_LOCAL / "gpu_projection"
    outputs.mkdir(parents=True, exist_ok=True)
    for i, level in enumerate(labels[:GPU_MAX_LEVELS]):
        if len(set(level.tolist())) <= 1:
            continue
        projected = estimate_block_kernel(training, level,
                                          device="cuda", batch_size=GPU_BATCH_SIZE)
        assert projected["backend"] == "cuda"
        out = outputs / f"empirical_W_level_{i:03d}.json"
        temp = out.with_suffix(".tmp")
        temp.write_text(json.dumps(projected, ensure_ascii=False), encoding="utf-8")
        os.replace(temp, out)
        print("GPU:", out.name, "блоков:", len(projected["block_mass"]),
              "ненулевых пар:", len(projected["entries"]))
        relation_totals = {}
        for item in projected["entries"]:
            rel = item["relation"]
            relation_totals[rel] = relation_totals.get(rel, 0) + item["edge_count"]
        print("  типы отношений по GPU-проекции:", relation_totals)
    persist_checkpoints()
else:
    print("GPU недоступен, CUDA-проекция пропущена; H1 nSBM остаётся CPU экспериментом.")

## 13. Как интерпретировать результаты

- Для ConceptNet-10k сравниваем MDL и прогнозирование трёх направленных многослойных моделей. В отчёте значения разделены по типам отношений.
- Устойчивость между инициализациями рассчитывается по ARI найденных разбиений; одинаковое количество блоков не означает совпадение состава.
- В синтетическом генераторе тип отношения теперь назначается независимым случайным процессом; отдельный калибровочный запуск измеряет ARI восстановления настоящих тонких и крупномасштабных блоков.
- Только QUICK/VALIDATION/FULL с заданными числами контролей. QUICK не обеспечивает достаточное разрешение p даже при положительном MDL; VALIDATION/FULL всё ещё требуют проверки ложных обнаружений на искусственных графах.
- CUDA применяется исключительно к эмпирическим ступенчатым ядрам W, отдельно для каждого типа связи; S/I не оцениваются. Graph-tool остаётся CPU.
- Отбор 10k вершин — relation-stratified BFS, и поэтому не является равномерной или несмещённой выборкой ConceptNet. Все выводы относятся к конкретному выбранному подграфу.

Для следующего опыта меняйте RUN_ID; старые 100k результаты сохраняются без изменений.

In [ ]:
# 13. Необязательная серия научной калибровки (дорогая; отдельно от H1 на ConceptNet)
if RUN_CALIBRATION:
    CAL_LOCAL = WORK_ROOT / "calibration" / f"{RUN_ID}-calibration"
    CAL_DRIVE = DRIVE_ROOT / "colab" / "hypothesis" / "calibration" / f"{RUN_ID}-calibration"
    if USE_DRIVE:
        CAL_DRIVE.mkdir(parents=True, exist_ok=True)
    calc = [str(ENV_PY), str(REPOSITORY / "scripts" / "calibrate_h1_synthetic.py"),
            "--output", str(CAL_LOCAL), "--simulations", str(CALIBRATION_SIMULATIONS),
            "--restarts", str(CALIBRATION_RESTARTS), "--nulls", str(CALIBRATION_NULLS),
            "--workers", "auto", "--resume"]
    # Restore each artificial graph's own checkpoint. No flat >10k-file Drive folders.
    if USE_DRIVE:
        for condition in ("flat", "nested"):
            for i in range(CALIBRATION_SIMULATIONS):
                local_one = CAL_LOCAL / condition / f"run-{i:03d}"
                drive_one = CAL_DRIVE / condition / f"run-{i:03d}"
                if (drive_one / "manifest.json").exists() and not local_one.exists():
                    restore_run(local_one, drive_one)

    def persist_calibration():
        copied = 0
        for condition in ("flat", "nested"):
            for i in range(CALIBRATION_SIMULATIONS):
                local_one = CAL_LOCAL / condition / f"run-{i:03d}"
                drive_one = CAL_DRIVE / condition / f"run-{i:03d}"
                copied += persist_run(local_one, drive_one)
        copied += copy_if_new(CAL_LOCAL / "calibration.json",
                              CAL_DRIVE / "calibration.json")
        return copied

    run_log = LOGS / f"{RUN_ID}-calibration.log"
    print("Это отдельный эксперимент на независимых искусственных графах.")
    print("Оптимизаций:", 2 * CALIBRATION_SIMULATIONS *
          3 * CALIBRATION_RESTARTS * (1 + 2 * CALIBRATION_NULLS))
    with run_log.open("a") as out:
        proc = subprocess.Popen(
            ["taskset", "-c", ",".join(map(str, CPU_IDS))] + calc
            if CPU_IDS and shutil.which("taskset") else calc,
            cwd=REPOSITORY, env=CPU_ENV, stdout=out, stderr=subprocess.STDOUT)
        try:
            while proc.poll() is None:
                time.sleep(CHECKPOINT_INTERVAL_SECONDS)
                print("Калибровка: новых файлов на Drive:", persist_calibration())
            rc = proc.wait()
        except KeyboardInterrupt:
            proc.terminate()
            try:
                proc.wait(timeout=20)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            persist_calibration()
            raise
        finally:
            persist_calibration()
    if rc != 0:
        raise RuntimeError("Калибровка прервалась: см. журнал; сохраните RUN_ID и повторите")
    print("Калибровка:", CAL_DRIVE / "calibration.json")
else:
    print("Калибровка не запускалась (RUN_CALIBRATION=False); итог H1 — предварительный.")